# Atividade Prática: Explorando Dados na Web com Python

**Notebook completo com códigos, respostas e interpretações no próprio arquivo.**

A atividade aborda requisições HTTP com `requests`, APIs JSON, tratamento de erros, arquivos binários, web scraping com BeautifulSoup e leitura de tabelas HTML com Pandas.


# Nível 1 – Básico
## Exercício 1.1 – Requisição GET

**Resposta:** Foi utilizada a API pública JSONPlaceholder para realizar uma requisição GET.


In [ ]:
import requests
import pandas as pd
import io
from bs4 import BeautifulSoup

url = 'https://jsonplaceholder.typicode.com/posts'

headers = {
    'User-Agent': 'ProjetoExplorandoDadosWeb/1.0'
}

resposta = requests.get(
    url,
    headers=headers,
    timeout=10
)

print('Status HTTP:', resposta.status_code)
print('URL final:', resposta.url)
print('Primeiro registro:')
print(resposta.json()[0])


## Exercício 1.2 – Parâmetros com `params`

**Resposta:** O parâmetro `userId=2` é enviado através do argumento `params`, fazendo a API retornar somente as publicações associadas ao usuário 2.


In [ ]:
params = {
    'userId': 2
}

resposta = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    params=params,
    headers=headers,
    timeout=10
)

print('Status HTTP:', resposta.status_code)
print('URL final:', resposta.url)
print('Quantidade de resultados:', len(resposta.json()))


## Exercício 1.3 – Cabeçalhos

**Resposta:** Foi utilizado um dicionário `headers` contendo um `User-Agent` personalizado que identifica o projeto.


In [ ]:
headers = {
    'User-Agent': 'ProjetoExplorandoDadosWeb/1.0'
}

resposta = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    headers=headers,
    timeout=10
)

print('User-Agent enviado:', headers['User-Agent'])
print('Status:', resposta.status_code)


## Exercício 1.4 – Status HTTP e URL final

**Resposta:** `status_code` mostra o resultado da requisição e `resposta.url` mostra a URL efetivamente montada pelo `requests`. O código 200 indica sucesso.


In [ ]:
params = {'userId': 2}

resposta = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    params=params,
    headers=headers,
    timeout=10
)

print('Código de status HTTP:', resposta.status_code)
print('URL final:', resposta.url)

if resposta.status_code == 200:
    print('Requisição realizada com sucesso!')


# Nível 2 – Intermediário
## Exercício 2.1 – Consulta de CEPs e DataFrame

**Resposta:** Três CEPs são consultados em um loop. O método `.json()` transforma cada resposta em um dicionário, que posteriormente é convertido em um DataFrame.


In [ ]:
ceps = ['01001000', '20040002', '30130000']

resultados = []

for cep in ceps:
    try:
        url_cep = f'https://viacep.com.br/ws/{cep}/json/'
        resposta = requests.get(url_cep, timeout=10)
        resposta.raise_for_status()

        dados_cep = resposta.json()
        dados_cep['cep_consultado'] = cep
        resultados.append(dados_cep)

    except requests.exceptions.RequestException as erro:
        print(f'Erro ao consultar o CEP {cep}: {erro}')

df_ceps = pd.DataFrame(resultados)

display(df_ceps)


## Exercício 2.2 – Função de download segura

**Resposta:** A função abaixo utiliza `try/except`, `raise_for_status()` e `timeout`. Assim, erros HTTP ou falhas de conexão são tratados sem encerrar o programa de forma inesperada.


In [ ]:
def download_seguro(url, nome_arquivo):
    try:
        resposta = requests.get(url, timeout=10)
        resposta.raise_for_status()

        with open(nome_arquivo, 'wb') as arquivo:
            arquivo.write(resposta.content)

        print(f'Download realizado com sucesso: {nome_arquivo}')

    except requests.exceptions.HTTPError as erro:
        print(f'Erro HTTP durante o download: {erro}')

    except requests.exceptions.RequestException as erro:
        print(f'Falha na requisição: {erro}')

    except OSError as erro:
        print(f'Erro ao salvar o arquivo: {erro}')


## Exercício 2.3 – Download de imagem

**Resposta:** A imagem aleatória do Picsum é baixada usando `resposta.content` e salva em modo binário (`wb`) com extensão `.jpg`.


In [ ]:
url_imagem = 'https://picsum.photos/400/400'

try:
    resposta = requests.get(url_imagem, timeout=10)
    resposta.raise_for_status()

    with open('imagem_aleatoria.jpg', 'wb') as arquivo:
        arquivo.write(resposta.content)

    print('Imagem salva como imagem_aleatoria.jpg')

except requests.exceptions.RequestException as erro:
    print(f'Erro ao baixar a imagem: {erro}')


# Nível 3 – Avançado
## Exercício 3.1 – Verificação do `robots.txt`

**Resposta:** Antes do scraping, o `robots.txt` do site é consultado. Isso permite verificar as regras publicadas pelo site para robôs/crawlers antes da coleta.


In [ ]:
robots_url = 'https://books.toscrape.com/robots.txt'

try:
    resposta_robots = requests.get(
        robots_url,
        headers=headers,
        timeout=10
    )
    resposta_robots.raise_for_status()

    print('Conteúdo do robots.txt:')
    print(resposta_robots.text)

except requests.exceptions.RequestException as erro:
    print(f'Não foi possível consultar o robots.txt: {erro}')


## Exercício 3.2 – Web scraping dos 5 primeiros livros

**Resposta:** A página do Books to Scrape é acessada com `requests`. Depois, o HTML é analisado com BeautifulSoup usando o parser `html.parser`. São extraídos o título e o preço dos cinco primeiros livros.


In [ ]:
url_livros = 'https://books.toscrape.com/'

try:
    resposta = requests.get(
        url_livros,
        headers=headers,
        timeout=10
    )
    resposta.raise_for_status()

    soup = BeautifulSoup(resposta.text, 'html.parser')

    livros = soup.select('article.product_pod')[:5]

    dados_livros = []

    for livro in livros:
        titulo = livro.find('h3').find('a')['title']
        preco = livro.select_one('.price_color').get_text(strip=True)

        dados_livros.append({
            'titulo': titulo,
            'preco': preco
        })

    df_livros = pd.DataFrame(dados_livros)

    display(df_livros)

except requests.exceptions.RequestException as erro:
    print(f'Erro ao acessar a página: {erro}')


## Exercício 3.3 – Salvar os livros em CSV

**Resposta:** O DataFrame criado no exercício anterior é salvo no arquivo `livros_5_primeiros.csv`.


In [ ]:
df_livros.to_csv(
    'livros_5_primeiros.csv',
    index=False,
    encoding='utf-8-sig'
)

print('Arquivo salvo como: livros_5_primeiros.csv')


## Exercício 3.4 – Tabela da Wikipedia com `read_html()` e `StringIO`

**Resposta:** Uma página da Wikipedia contendo uma tabela é carregada com `requests`. O HTML é colocado em `io.StringIO()` e passado para `pd.read_html()`, que identifica as tabelas e as transforma em DataFrames sem utilizar BeautifulSoup.


In [ ]:
url_wikipedia = 'https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_por_popula%C3%A7%C3%A3o'

try:
    resposta_wiki = requests.get(
        url_wikipedia,
        headers=headers,
        timeout=10
    )
    resposta_wiki.raise_for_status()

    tabelas = pd.read_html(io.StringIO(resposta_wiki.text))

    print('Quantidade de tabelas encontradas:', len(tabelas))

    # Exibe a primeira tabela encontrada
    df_wikipedia = tabelas[0]
    display(df_wikipedia.head())

except requests.exceptions.RequestException as erro:
    print(f'Erro ao acessar a Wikipedia: {erro}')
except ValueError as erro:
    print(f'Não foi possível interpretar as tabelas HTML: {erro}')


# Conclusão

A atividade apresentou as principais etapas para trabalhar com dados disponíveis na Web usando Python:

- `requests.get()` para realizar requisições HTTP;
- `params` para enviar parâmetros na URL;
- `headers` para personalizar o User-Agent;
- `status_code` e `raise_for_status()` para verificar erros;
- `.json()` para transformar respostas JSON em estruturas Python;
- `DataFrame` para organizar dados obtidos de APIs;
- `response.content` e modo `wb` para arquivos binários;
- `BeautifulSoup` para localizar informações em páginas HTML;
- `robots.txt` como etapa de verificação antes de scraping;
- `pd.read_html()` + `io.StringIO()` para extrair tabelas HTML diretamente para DataFrames.

**Observação:** os códigos utilizam `timeout=10` nas requisições para evitar que uma conexão fique aguardando indefinidamente. O scraping deve ser realizado de forma responsável, respeitando as regras do site e evitando requisições excessivas.
